In [ ]:
import os
import numpy as np
import json
import random
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm
import shutil

annotation_path = "SPair-71k/PairAnnotation4x/test"
image_path = "SPair-71k/JPEGImages4x"
annotation_files = os.listdir(annotation_path)
annotation_files = [os.path.join(annotation_path, f) for f in annotation_files]
save_directory = "SemCorVQA/test"


CIRCLE_RADIUS = 30
THICKNESS = 6
FONT_SIZE = 48
MAX_SAMPLES = 1_000_0000

In [6]:

categorization = {"aeroplane":
                    {"unique": [0, 1, 3, 23, 24],
                    "two": [16, 17, 18, 19, 6, 7, 4, 5, 21, 22, 8, 9, 10, 11, 12, 13, 14, 15],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15],
                    "no-name": [2, 12, 13, 16, 17, 18, 19, 21, 22, 24],
                    "partial/others": []},

                  "bicycle":
                    {"unique": [4, 8, 13],
                    "two": [2,3,9,10, 0,1],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [0, 2,3, 1, 9, 10, 13],
                    "no-name": [4, 8, 5,6,7],
                    "partial/others": [5,6,7]},

                    "bird":
                    {"unique": [3, 0, 16, 6, 9],
                    "two": [10,11,14,15,5, 7,8, 4],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [0, 16, 3, 1, 2, 11, 10, 7, 8],
                    "no-name": [14, 15, 9, 4, 5, 6],
                    "partial/others": []},

                    "boat":
                    {"unique": [],
                    "two": [],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [],
                    "no-name": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
                    "partial/others": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]},


                    "bottle":
                    {"unique": [],
                    "two": [0, 1, 8, 9],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [0,1,8,9],
                    "no-name": [2,3,4,5,6,7],
                    "partial/others": [2,3,4,5,6,7]},

                    "bus":
                    {"unique": [4,7],
                    "two": [2,3, 0, 1],
                    "three": [],
                    "four": [24,21,11,14],
                    "four+": [],
                    "named": [4,7,2,3,24,11,14, 0, 1],
                    "no-name": [20,22,23,25,26, 27, 28,29, 5,6, 10, 12, 13, 15,16,17, 18, 19, 21],
                    "partial/others": [20,22,23,25]},


                    "car":
                    {"unique": [4,5,8,9],
                    "two": [2,3, 0, 1, 6,7],
                    "three": [],
                    "four": [11,14,21,24],
                    "four+": [],
                    "named": [2,3,4,5,1,0,14,11,21,24,6,7,8,9],
                    "no-name": [25,23,22,20,10,12,13,15, 16,17,18,19,28,26,27, 29],
                    "partial/others": [25,23,22,20,10,12,13,15, 16,17,18,19,28,26,27,29]},

                    "cat":
                    {"unique": [13, 8, 14],
                    "two": [0,12,3,4,5],
                    "three": [],
                    "four": [9, 10, 11, 12],
                    "four+": [],
                    "named": [2,3,4,5,8, 9, 10, 11, 12,13],
                    "no-name": [14,0,1,6,7, 14],
                    "partial/others": [6,7]},

                    "chair":
                    {"unique": [],
                    "two": [8,9,10,11],
                    "three": [],
                    "four": [4,5,6,7],
                    "four+": [],
                    "named": [8,9,10,11,4,5,6,7],
                    "no-name": [0,1,2,3],
                    "partial/others": [0,1,2,3]},

                    "cow":
                    {"unique": [9,13,14],
                    "two": [19,20,4,5, 0,1,2,3,15, 16],
                    "three": [],
                    "four": [9,10,11,12],
                    "four+": [],
                    "named": [19,20,4,5,13,9,10,11,12,2,3,8],
                    "no-name": [0,1,14,6,7, 15,16],
                    "partial/others": []},


                    "dog":
                    {"unique": [7,8, 13,14,15,6],
                    "two": [2,3,4,5, 0, 1],
                    "three": [],
                    "four": [9,10,11,12],
                    "four+": [],
                    "named": [7,8,13,2,3,4,5, 9,10,11,12, 15,6],
                    "no-name": [14, 0, 1],
                    "partial/others": []},

                    "horse":
                    {"unique": [9,8,14,15],
                    "two": [2,3,4,5,6,7,0,1],
                    "three": [],
                    "four": [10, 11, 12, 13, 16, 17, 18, 19],
                    "four+": [],
                    "named": [2,3,4,5,6,7,9,8, 14, 10,11,12,13],
                    "no-name": [0,1, 16, 17, 15, 18, 19],
                    "partial/others": []},

                    "motorbike":
                    {"unique": [4,6,5,7,8,9, 10],
                    "two": [2,3, 0,1],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [4,11,12,2,3,0,1,6,5,7,8, 10],
                    "no-name": [9],
                    "partial/others": [11,2]},

                    "person":
                    {"unique": [4,7,5,6],
                    "two": [0,1,2,3,8,9,12,13,10,11,19,18,17,16,15,14],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [0,1,2,3,4,5,7,8,9,10,11,12,13,14,15,16,17,18,19],
                    "no-name": [6],
                    "partial/others": []},

                    "pottedplant":
                    {"unique": [],
                    "two": [],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [],
                    "no-name": [0,1,2,3,4,5,6,7,8],
                    "partial/others": [0,1,2,3,4,5,6,7,8]},

                    "sheep":
                    {"unique": [13,14,8],
                    "two": [2,3,4,5,0,1, 19, 20],
                    "three": [],
                    "four": [9,10,11,12,15,16,17,18],
                    "four+": [],
                    "named": [8,2,3,4,5, 19, 20,13,9,10,11,12],
                    "no-name": [6,7,0,1,14,15,16,17,18],
                    "partial/others": [6,7]},



                    "train":
                    {"unique": [],
                    "two": [16,17,1,2,3,8,9,19,11,12,13,14,15,16],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [16,17],
                    "no-name": [0,1,2,3,8,9,19,11,12,13,14,15,4,6,5,7],
                    "partial/others": [4,6,5,7]},

                    "tvmonitor":
                    {"unique": [],
                    "two": [],
                    "three": [],
                    "four": [],
                    "four+": [],
                    "named": [],
                    "no-name": [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
                    "partial/others": [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]}
                  }

In [7]:
if os.path.exists(save_directory):
    shutil.rmtree(save_directory)
os.makedirs(save_directory)
os.makedirs(os.path.join(save_directory, "images"))


# All tag keys we want to check membership for
TAG_KEYS = ["unique", "two", "four", "named", "no-name", "partial/others"]

def get_kp_tags(category, kp_id_str):
    """Return list of tags that this keypoint belongs to for this category."""
    cat = categorization.get(category, {})
    kp_id = int(kp_id_str)
    return [tag for tag in TAG_KEYS if kp_id in cat.get(tag, [])]

# ── HELPERS ───────────────────────────────────────────────────────────────────
def skip_sample(data):
    frac = 20
    src_image_height = data["src_imsize"][1]
    for i in range(len(data["src_kps"])):
        for j in range(i + 1, len(data["src_kps"])):
            if np.linalg.norm(np.array(data["src_kps"][i]) - np.array(data["src_kps"][j])) < src_image_height / frac:
                return True
    tgt_image_height = data["trg_imsize"][1]
    for i in range(len(data["trg_kps"])):
        for j in range(i + 1, len(data["trg_kps"])):
            if np.linalg.norm(np.array(data["trg_kps"][i]) - np.array(data["trg_kps"][j])) < tgt_image_height / frac:
                return True
    return False

def draw_annotation(base_image, coordinates, texts):
    image = base_image.copy()
    r = CIRCLE_RADIUS
    font_size = FONT_SIZE
    draw = ImageDraw.Draw(image)
    font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', font_size)
    for (x, y), text in zip(coordinates, texts):
        draw.ellipse([(x - r, y - r), (x + r, y + r)], outline=(255, 0, 255), width=THICKNESS)
        bbox = draw.textbbox((0, 0), text, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        text_offset_y = bbox[1]
        tx = x - tw // 2
        ty = y - r - th - 4 if y > th + 10 else y + r + 4
        draw.rectangle([(tx - 2, ty + text_offset_y - 2), (tx + tw + 2, ty + text_offset_y + th + 2)], fill='black')
        draw.text((tx, ty), text, fill='white', font=font)
    return image

# ── MAIN ──────────────────────────────────────────────────────────────────────
skipped = 0
global_index = 0
annotations = []

for file in tqdm(annotation_files):
    with open(file, "r") as f:
        data = json.load(f)

    if skip_sample(data):
        skipped += 1
        continue

    category = data["category"]
    src_coordinates = data["src_kps"]
    tgt_coordinates = data["trg_kps"]

    src_image = Image.open(os.path.join(image_path, category, data["src_imname"]))
    tgt_image = Image.open(os.path.join(image_path, category, data["trg_imname"]))

    # Try EVERY annotated point as REF
    for i, kid in enumerate(data["kps_ids"]):
        ref = src_coordinates[i]
        corresponding_tgt = tgt_coordinates[i]

        tgt_distractors = [tgt_coordinates[j] for j in range(len(tgt_coordinates)) if j != i]
        if len(tgt_distractors) < 3:
            continue

        tgt_points_sampled = random.sample(tgt_distractors, 3)
        tgt_points_final = [corresponding_tgt] + tgt_points_sampled

        options = ["A", "B", "C", "D"]
        random.seed(global_index)
        random.shuffle(options)
        correct_option = options[0]

        modified_src_image = draw_annotation(src_image, [ref], ["REF"])
        modified_tgt_image = draw_annotation(tgt_image, tgt_points_final, options)

        save_path1 = os.path.join(save_directory, "images", f"{global_index:06d}_ref.jpg")
        save_path2 = os.path.join(save_directory, "images", f"{global_index:06d}_tgt.jpg")

        modified_src_image.save(save_path1)
        modified_tgt_image.save(save_path2)

        annotations.append({
            "ref_image_path": save_path1,
            "tgt_image_path": save_path2,
            "options": options,
            "answer": correct_option,
            "src_index": i,
            "kps_id": kid,
            "kps_tags": get_kp_tags(category, kid),   # e.g. ["named", "unique"]
            "category": category,
            "ref_coordinate": ref,
            "corresponding_tgt": corresponding_tgt,
            "tgt_points_sampled": tgt_points_sampled,
            "og_src_image": os.path.join(image_path, category, data["src_imname"]),
            "og_tgt_image": os.path.join(image_path, category, data["trg_imname"]),
            "original_data": data,
        })

        global_index += 1

        if global_index % 100 == 0:
            with open(os.path.join(save_directory, "annotations.json"), "w") as f:
                json.dump(annotations, f)

        if global_index >= MAX_SAMPLES:
            break

    if global_index >= MAX_SAMPLES:
        break

with open(os.path.join(save_directory, "annotations.json"), "w") as f:
    json.dump(annotations, f)

print(f"Skipped {skipped} samples")
print(f"Total samples generated: {len(annotations)}")

  5%|▍         | 559/12234 [00:18<06:21, 30.61it/s]

Skipped 346 samples
Total samples generated: 1000


In [8]:
import json

with open("SemCorVQA/test/annotations.json", "r") as f:
    data = json.load(f)

data[0]

{'ref_image_path': 'SemCorVQA/test/images/000000_ref.jpg',
 'tgt_image_path': 'SemCorVQA/test/images/000000_tgt.jpg',
 'options': ['C', 'A', 'B', 'D'],
 'answer': 'C',
 'src_index': 0,
 'kps_id': '6',
 'kps_tags': ['unique', 'named'],
 'category': 'motorbike',
 'ref_coordinate': [1368, 436],
 'corresponding_tgt': [1224, 1092],
 'tgt_points_sampled': [[444, 500], [556, 820], [1292, 820]],
 'og_src_image': 'SPair-71k/JPEGImages4x/motorbike/2011_001022.jpg',
 'og_tgt_image': 'SPair-71k/JPEGImages4x/motorbike/2011_000060.jpg',
 'original_data': {'pair_id': 8867,
  'filename': '008867-2011_001022-2011_000060:motorbike',
  'src_imname': '2011_001022.jpg',
  'trg_imname': '2011_000060.jpg',
  'src_imsize': [2000, 1332, 3],
  'trg_imsize': [2000, 1500, 3],
  'src_bndbox': [256, 4, 1580, 616],
  'trg_bndbox': [416, 156, 1452, 1384],
  'category': 'motorbike',
  'src_pose': 'Right',
  'trg_pose': 'Unspecified',
  'src_kps': [[1368, 436], [544, 420], [420, 468], [1476, 264]],
  'trg_kps': [[1224,